# **`Blinkit Data`**

## **Import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv('/content/blinkit_dataset.csv')
display(df.head())

In [ ]:
df.tail()

In [ ]:
# Dataset shape
print("Dataset Shape:")
display(df.shape)

# Column names
print("\nColumn Names:")
display(df.columns.tolist())

# First 5 rows
print("\nFirst 5 Rows:")
display(df.head())

# Last 5 rows
print("\nLast 5 Rows:")
display(df.tail())

# Data types
print("\nData Types:")
display(df.dtypes)

# Dataset information
print("\nDataset Information:")
df.info()

# Missing values
print("\nMissing Values:")
display(df.isnull().sum())

# Unique values
print("\nUnique Values:")
display(df.nunique())

In [ ]:
# Create a working copy
data = df.copy()

print("Original Shape:", data.shape)

## **Data Cleaning**

**1. Check duplicates**

In [ ]:
print("Duplicate Rows:", data.duplicated().sum())

In [ ]:
data = data.drop_duplicates()

print("Shape After Removing Duplicates:", data.shape)

**2. Handle missing values**

In [ ]:
print("Missing Values Before Cleaning:")
display(data.isnull().sum())

In [ ]:
data["offer_type"] = data["offer_type"].fillna("No Offer")

In [ ]:
print("\nMissing Values After Cleaning:")
display(data.isnull().sum())

**3. Convert dates**

In [ ]:
data["date_added"] = pd.to_datetime(data["date_added"])
data["expiry_date"] = pd.to_datetime(data["expiry_date"])

display(data[["date_added", "expiry_date"]].dtypes)

**4. Check invalid values**

In [ ]:
display("Price <= 0:", (data["price"] <= 0).sum())
display("Final Price <= 0:", (data["final_price"] <= 0).sum())
display("Rating outside 0-5:", ((data["rating"] < 0) | (data["rating"] > 5)).sum())
display("Discount outside 0-100:", ((data["discount_pct"] < 0) | (data["discount_pct"] > 100)).sum())
display("Stock < 0:", (data["stock"] < 0).sum())
display("Sold Quantity < 0:", (data["sold_quantity"] < 0).sum())
display("Delivery Time < 0:", (data["delivery_time_min"] < 0).sum())

**5. Final cleaning check**

In [ ]:
display("\n========== CLEANED DATASET ==========")

display("Dataset Shape:", data.shape)

display("\nMissing Values:")
display(data.isnull().sum())

display("\nDuplicate Rows:", data.duplicated().sum())

display("\nData Types:")
display(data.dtypes)

**6. Save cleaned dataset**

In [ ]:
data.to_csv("blinkit_cleaned.csv", index=False)

display("Cleaned dataset saved successfully!")

## **EDA**

**Sales by Category**

In [ ]:
plt.figure(figsize=(12, 6))

category_sales = (
    data.groupby("category")["sold_quantity"]
    .sum()
    .sort_values(ascending=True)
)

plt.barh(
    category_sales.index,
    category_sales.values
)

plt.title("Total Sales by Category", fontsize=16, fontweight="bold")
plt.xlabel("Total Sold Quantity")
plt.ylabel("Category")

plt.tight_layout()
plt.show()

**Sales by City**

In [ ]:
city_sales = (
    data.groupby("city")["sold_quantity"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 6))

sns.barplot(
    x=city_sales.index,
    y=city_sales.values
)

plt.title("Total Sales by City", fontsize=16, fontweight="bold")
plt.xlabel("City")
plt.ylabel("Total Sold Quantity")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

**Price vs Sold Quantity**

In [ ]:
plt.figure(figsize=(20, 15))

sns.regplot(
    data=data,
    x="final_price",
    y="sold_quantity",
    scatter_kws={"alpha": 0.4},
    line_kws={}
)

plt.title(
    "Final Price vs Sold Quantity",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Final Price")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

**Demand Index vs Sold Quantity**

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=data,
    x="demand_index",
    y="sold_quantity",
    alpha=0.6
)

plt.title(
    "Demand Index vs Sold Quantity",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Demand Index")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

**Correlation Heatmap**

In [ ]:
numeric_data = data.select_dtypes(include=np.number)

correlation = numeric_data.corr()

plt.figure(figsize=(14, 10))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title(
    "Correlation Heatmap",
    fontsize=20,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

**Interactive Version**

In [ ]:
import plotly.express as px

fig = px.scatter(
    data,
    x="final_price",
    y="sold_quantity",
    color="category",
    hover_data=[
        "product_name",
        "brand",
        "rating",
        "discount_pct",
        "demand_index"
    ],
    title="Interactive Final Price vs Sold Quantity"
)

fig.show()

**Final Price × Rating × Sold Quantity**

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    data,
    x="final_price",
    y="rating",
    z="sold_quantity",
    color="category",
    hover_name="product_name",
    title="3D Price, Rating and Sales Analysis"
)

fig.show()

**Sales by City**

In [ ]:
city_sales = (
    data.groupby("city")["sold_quantity"]
    .sum()
    .sort_values(ascending=False)
)

print("Sales by City:")
display(city_sales)

In [ ]:
plt.figure(figsize=(12, 6))

ax = sns.barplot(
    x=city_sales.values,
    y=city_sales.index
)

plt.title(
    "Total Sales by City",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Total Sold Quantity")
plt.ylabel("City")

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

city_sales_df = (
    data.groupby("city", as_index=False)["sold_quantity"]
    .sum()
    .sort_values("sold_quantity", ascending=False)
)

fig = px.bar(
    city_sales_df,
    x="city",
    y="sold_quantity",
    title="Interactive Total Sales by City",
    labels={
        "city": "City",
        "sold_quantity": "Total Sold Quantity"
    },
    text="sold_quantity"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white"
)

fig.show()

**Top 10 Brands by Sales**

In [ ]:
top_brands = (
    data.groupby("brand")["sold_quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

display("Top 10 Brands by Sales:")
display(top_brands)

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    x=top_brands.values,
    y=top_brands.index
)

plt.title(
    "Top 10 Brands by Total Sales",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Total Sold Quantity")
plt.ylabel("Brand")

plt.tight_layout()
plt.show()

**Delivery Status**

In [ ]:
display("Delivery Status Counts:")
display(data["delivery_status"].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.countplot(
    data=data,
    x="delivery_status"
)

plt.title(
    "Distribution of Delivery Status",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Delivery Status")
plt.ylabel("Number of Products/Orders")

for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.tight_layout()
plt.show()

**Percentage distribution**

In [ ]:
delivery_percentage = (
    data["delivery_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display("Delivery Status Percentage:")
display(delivery_percentage)

In [ ]:
delivery_counts = (
    data["delivery_status"]
    .value_counts()
    .reset_index()
)

delivery_counts.columns = ["delivery_status", "count"]

fig = px.bar(
    delivery_counts,
    x="delivery_status",
    y="count",
    text="count",
    title="Interactive Delivery Status Distribution"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Delivery Status",
    yaxis_title="Count"
)

fig.show()

**Offer Type Distribution**

In [ ]:
display("Offer Type Counts:")
display(data["offer_type"].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.countplot(
    data=data,
    x="offer_type"
)

plt.title(
    "Distribution of Offer Types",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Offer Type")
plt.ylabel("Number of Products")

plt.xticks(rotation=30)

for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
offer_counts = (
    data["offer_type"]
    .value_counts()
    .reset_index()
)

offer_counts.columns = ["offer_type", "count"]

fig = px.bar(
    offer_counts,
    x="offer_type",
    y="count",
    text="count",
    title="Interactive Offer Type Distribution"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Offer Type",
    yaxis_title="Count"
)

fig.show()

**Price Distribution**

In [ ]:
display("Price Statistics:")
display(data["price"].describe())

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=data,
    x="price",
    bins=30,
    kde=True
)

plt.title(
    "Product Price Distribution",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Price")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

sns.boxplot(
    x=data["price"]
)

plt.title(
    "Product Price Box Plot",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Price")

plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

fig = px.histogram(
    data,
    x="price",
    nbins=30,
    marginal="box",
    title="Interactive Product Price Distribution",
    labels={"price": "Price"}
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Rating Distribution**

In [ ]:
display("Rating Statistics:")
display(data["rating"].describe())

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=data,
    x="rating",
    bins=20,
    kde=True
)

plt.title(
    "Customer Rating Distribution",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Rating")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(
    x=data["rating"]
)

plt.title(
    "Customer Rating Box Plot",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Rating")

plt.tight_layout()
plt.show()

In [ ]:
fig = px.histogram(
    data,
    x="rating",
    nbins=20,
    marginal="box",
    title="Interactive Customer Rating Distribution",
    labels={"rating": "Customer Rating"}
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Sold Quantity Distribution**

In [ ]:
display("Sold Quantity Statistics:")
display(data["sold_quantity"].describe())

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=data,
    x="sold_quantity",
    bins=30,
    kde=True
)

plt.title(
    "Sold Quantity Distribution",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Sold Quantity")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

sns.boxplot(
    x=data["sold_quantity"]
)

plt.title(
    "Sold Quantity Box Plot",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Sold Quantity")

plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

fig = px.histogram(
    data,
    x="sold_quantity",
    nbins=30,
    marginal="box",
    title="Interactive Sold Quantity Distribution",
    labels={"sold_quantity": "Sold Quantity"}
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Demand Index Distribution**

In [ ]:
display("Demand Index Statistics:")
display(data["demand_index"].describe())

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=data,
    x="demand_index",
    bins=30,
    kde=True
)

plt.title(
    "Demand Index Distribution",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Demand Index")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

sns.boxplot(
    x=data["demand_index"]
)

plt.title(
    "Demand Index Box Plot",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Demand Index")

plt.tight_layout()
plt.show()

In [ ]:
fig = px.histogram(
    data,
    x="demand_index",
    nbins=30,
    marginal="box",
    title="Interactive Demand Index Distribution",
    labels={"demand_index": "Demand Index"}
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Discount Distribution**

In [ ]:
display("Discount Statistics:")
display(data["discount_pct"].describe())

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=data,
    x="discount_pct",
    bins=25,
    kde=True
)

plt.title(
    "Discount Percentage Distribution",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Discount (%)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

sns.boxplot(
    x=data["discount_pct"]
)

plt.title(
    "Discount Percentage Box Plot",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Discount (%)")

plt.tight_layout()
plt.show()

**Price vs Sold Quantity**

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=data,
    x="price",
    y="sold_quantity",
    alpha=0.5
)

plt.title(
    "Price vs Sold Quantity",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Price")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.regplot(
    data=data,
    x="price",
    y="sold_quantity",
    scatter_kws={"alpha": 0.35},
    line_kws={}
)

plt.title(
    "Price vs Sold Quantity with Trend Line",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Price")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

**Calculate correlation**

In [ ]:
price_sales_corr = data["price"].corr(data["sold_quantity"])

display("Correlation between Price and Sold Quantity:",
      round(price_sales_corr, 4))

In [ ]:
import plotly.express as px

fig = px.scatter(
    data,
    x="price",
    y="sold_quantity",
    color="category",
    hover_data=[
        "product_name",
        "brand",
        "rating",
        "discount_pct",
        "demand_index"
    ],
    title="Interactive Price vs Sold Quantity"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Price",
    yaxis_title="Sold Quantity"
)

fig.show()

**Discount vs Sold Quantity**

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=data,
    x="discount_pct",
    y="sold_quantity",
    alpha=0.5
)

plt.title(
    "Discount Percentage vs Sold Quantity",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Discount (%)")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

**Correlation**

In [ ]:
discount_sales_corr = data["discount_pct"].corr(
    data["sold_quantity"]
)

display(
    "Correlation between Discount and Sold Quantity:",
    round(discount_sales_corr, 4)
)

In [ ]:
fig = px.scatter(
    data,
    x="discount_pct",
    y="sold_quantity",
    color="category",
    hover_data=["product_name", "brand", "price"],
    title="Interactive Discount vs Sold Quantity"
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

**Rating vs Sold Quantity**

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=data,
    x="rating",
    y="sold_quantity",
    alpha=0.5
)

plt.title(
    "Customer Rating vs Sold Quantity",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Rating")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

**Correlation**

In [ ]:
rating_sales_corr = data["rating"].corr(
    data["sold_quantity"]
)

display(
    "Correlation between Rating and Sold Quantity:",
    round(rating_sales_corr, 4)
)

**Demand Index vs Sold Quantity**

In [ ]:
plt.figure(figsize=(10, 6))

sns.regplot(
    data=data,
    x="demand_index",
    y="sold_quantity",
    scatter_kws={"alpha": 0.35},
    line_kws={}
)

plt.title(
    "Demand Index vs Sold Quantity with Trend",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Demand Index")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

**Correlation**

In [ ]:
demand_sales_corr = data["demand_index"].corr(
    data["sold_quantity"]
)

display(
    "Correlation between Demand Index and Sold Quantity:",
    round(demand_sales_corr, 4)
)

**Stock vs Sold Quantity**

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=data,
    x="stock",
    y="sold_quantity",
    alpha=0.5
)

plt.title(
    "Stock vs Sold Quantity",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Stock")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.regplot(
    data=data,
    x="stock",
    y="sold_quantity",
    scatter_kws={"alpha": 0.35},
    line_kws={}
)

plt.title(
    "Stock vs Sold Quantity with Trend",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Stock")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

**Correlation**

In [ ]:
stock_sales_corr = data["stock"].corr(
    data["sold_quantity"]
)

display(
    "Correlation between Stock and Sold Quantity:",
    round(stock_sales_corr, 4)
)

**Delivery Time vs Sold Quantity**

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=data,
    x="delivery_time_min",
    y="sold_quantity",
    alpha=0.5
)

plt.title(
    "Delivery Time vs Sold Quantity",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.regplot(
    data=data,
    x="delivery_time_min",
    y="sold_quantity",
    scatter_kws={"alpha": 0.35},
    line_kws={}
)

plt.title(
    "Delivery Time vs Sold Quantity with Trend",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Sold Quantity")

plt.tight_layout()
plt.show()

**Correlation**

In [ ]:
delivery_sales_corr = data["delivery_time_min"].corr(
    data["sold_quantity"]
)

display(
    "Correlation between Delivery Time and Sold Quantity:",
    round(delivery_sales_corr, 4)
)

**Complete Correlation Heatmap**

In [ ]:
numeric_data = data.select_dtypes(include=np.number)

correlation_matrix = numeric_data.corr()

plt.figure(figsize=(14, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title(
    "Correlation Heatmap of Numerical Features",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

**Find correlations specifically with `sold_quantity`**

In [ ]:
target_correlation = (
    correlation_matrix["sold_quantity"]
    .sort_values(ascending=False)
)

display("Correlation with Sold Quantity:")
display(target_correlation)

**Sales Trend Over Time**

In [ ]:
data["month"] = data["date_added"].dt.to_period("M").astype(str)

In [ ]:
monthly_sales = (
    data.groupby("month")["sold_quantity"]
    .sum()
    .reset_index()
)

display(monthly_sales.head())

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=monthly_sales,
    x="month",
    y="sold_quantity",
    marker="o"
)

plt.title(
    "Monthly Sales Trend",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Month")
plt.ylabel("Total Sold Quantity")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig = px.line(
    monthly_sales,
    x="month",
    y="sold_quantity",
    markers=True,
    title="Interactive Monthly Sales Trend",
    labels={
        "month": "Month",
        "sold_quantity": "Total Sold Quantity"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

### **Advanced Plotly Analysis**

**Interactive Sales by Category**

In [ ]:
import plotly.express as px

category_sales_df = (
    data.groupby("category", as_index=False)["sold_quantity"]
    .sum()
    .sort_values("sold_quantity", ascending=False)
)

fig = px.bar(
    category_sales_df,
    x="category",
    y="sold_quantity",
    color="category",
    text="sold_quantity",
    title="Interactive Sales by Category"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Category",
    yaxis_title="Total Sold Quantity",
    showlegend=False
)

fig.show()

**Interactive Price vs Sales**

In [ ]:
fig = px.scatter(
    data,
    x="price",
    y="sold_quantity",
    color="category",
    size="demand_index",
    hover_name="product_name",
    hover_data=[
        "brand",
        "rating",
        "discount_pct",
        "stock"
    ],
    title="Price vs Sold Quantity — Interactive Analysis"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Price",
    yaxis_title="Sold Quantity"
)

fig.show()

**Discount vs Sales**

In [ ]:
fig = px.scatter(
    data,
    x="discount_pct",
    y="sold_quantity",
    color="category",
    size="rating",
    hover_name="product_name",
    hover_data=[
        "price",
        "final_price",
        "brand",
        "demand_index"
    ],
    title="Discount vs Sold Quantity — Interactive Analysis"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Discount (%)",
    yaxis_title="Sold Quantity"
)

fig.show()

**Demand vs Sales**

In [ ]:
fig = px.scatter(
    data,
    x="demand_index",
    y="sold_quantity",
    color="category",
    size="final_price",
    hover_name="product_name",
    hover_data=[
        "brand",
        "rating",
        "discount_pct",
        "stock"
    ],
    title="Demand Index vs Sold Quantity — Interactive Analysis"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Demand Index",
    yaxis_title="Sold Quantity"
)

fig.show()

**Price × Rating × Sold Quantity**

In [ ]:
fig = px.scatter_3d(
    data,
    x="price",
    y="rating",
    z="sold_quantity",
    color="category",
    size="demand_index",
    hover_name="product_name",
    hover_data=[
        "brand",
        "discount_pct",
        "stock"
    ],
    title="Price, Rating and Sold Quantity"
)

fig.update_layout(
    template="plotly_white",
    scene=dict(
        xaxis_title="Price",
        yaxis_title="Rating",
        zaxis_title="Sold Quantity"
    )
)

fig.show()

**Discount × Demand Index × Sold Quantity**

In [ ]:
fig = px.scatter_3d(
    data,
    x="discount_pct",
    y="demand_index",
    z="sold_quantity",
    color="category",
    size="rating",
    hover_name="product_name",
    title="Discount, Demand and Sales"
)

fig.update_layout(
    template="plotly_white",
    scene=dict(
        xaxis_title="Discount (%)",
        yaxis_title="Demand Index",
        zaxis_title="Sold Quantity"
    )
)

fig.show()

## **Feature Engineering**

**ML dataset**

In [ ]:
model_df = data.copy()

print("Initial Shape:", model_df.shape)

In [ ]:
model_df["year"] = model_df["date_added"].dt.year
model_df["month"] = model_df["date_added"].dt.month
model_df["day"] = model_df["date_added"].dt.day
model_df["day_of_week"] = model_df["date_added"].dt.dayofweek

In [ ]:
display(
    model_df[
        ["date_added", "year", "month", "day", "day_of_week"]
    ].head()
)

**Create weekend feature**

In [ ]:
model_df["is_weekend"] = (
    model_df["day_of_week"] >= 5
).astype(int)

**Calculate discount amount**

In [ ]:
model_df["discount_amount"] = (
    model_df["price"] * model_df["discount_pct"] / 100
)

**Calculate price after discount**

In [ ]:
model_df["calculated_final_price"] = (
    model_df["price"] - model_df["discount_amount"]
)

In [ ]:
display(
    model_df[
        [
            "price",
            "discount_pct",
            "discount_amount",
            "final_price",
            "calculated_final_price"
        ]
    ].head()
)

**Create inventory utilization**

In [ ]:
model_df["inventory_utilization"] = (
    model_df["sold_quantity"] /
    (model_df["stock"] + model_df["sold_quantity"])
)

*Convert to percentage:*

In [ ]:
model_df["inventory_utilization_pct"] = (
    model_df["inventory_utilization"] * 100
)

**Create review engagement feature**

In [ ]:
model_df["review_engagement"] = (
    model_df["num_reviews"] /
    (model_df["sold_quantity"] + 1)
)

**Create product age**

In [ ]:
reference_date = model_df["date_added"].max()

model_df["product_age_days"] = (
    reference_date - model_df["date_added"]
).dt.days

**Create expiry duration**

In [ ]:
model_df["days_until_expiry"] = (
    model_df["expiry_date"] -
    model_df["date_added"]
).dt.days

**Create price-per-gram**

In [ ]:
model_df["price_per_gram"] = (
    model_df["final_price"] /
    model_df["weight_g"].replace(0, np.nan)
)

**Create price-per-100g**

In [ ]:
model_df["price_per_100g"] = (
    model_df["final_price"] /
    model_df["weight_g"].replace(0, np.nan)
) * 100

**Check the engineered features**

In [ ]:
new_features = [
    "year",
    "month",
    "day",
    "day_of_week",
    "is_weekend",
    "discount_amount",
    "calculated_final_price",
    "product_age_days",
    "days_until_expiry",
    "price_per_gram",
    "price_per_100g"
]

display("New Features:")
display(model_df[new_features].head())

**Check missing values created by engineering**

In [ ]:
display("Missing Values After Feature Engineering:")
display(
    model_df[new_features].isnull().sum()
)

**Check infinite values**

In [ ]:
display(
    "Infinite Values:",
    np.isinf(
        model_df.select_dtypes(include=np.number)
    ).sum().sum()
)

In [ ]:
model_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

**Final feature-engineered dataset**

In [ ]:
display("Feature-Engineered Dataset Shape:")
display(model_df.shape)

display("\nColumns:")
display(model_df.columns.tolist())

### **ML Preparation**

**Define Target & Features**

In [ ]:
# ==========================================
# STEP 5: MACHINE LEARNING PREPARATION
# ==========================================

target = "sold_quantity"

features = [
    "price",
    "final_price",
    "discount_pct",
    "rating",
    "num_reviews",
    "delivery_time_min",
    "stock",
    "profit_margin_pct",
    "weight_g",
    "shelf_life_days",
    "demand_index",
    "is_organic",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "discount_amount",
    "product_age_days",
    "days_until_expiry",
    "price_per_gram",
    "price_per_100g"
]

X = model_df[features].copy()
y = model_df[target].copy()

print("========================================")
print("        ML DATA PREPARATION")
print("========================================")

print("\nX Shape:", X.shape)
print("y Shape:", y.shape)

print("\nTarget Variable:")
print(target)

print("\nFeature Count:")
print(len(features))

print("\nFeatures:")
print(features)

**Check Data Types**

In [ ]:
display("\n========== DATA TYPES ==========")
display(X.dtypes)

**Separate Numerical & Categorical Features**

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32", "bool"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\n========== NUMERICAL FEATURES ==========")
print(numeric_features)

print("\nNumber of Numerical Features:", len(numeric_features))

print("\n========== CATEGORICAL FEATURES ==========")
print(categorical_features)

print("\nNumber of Categorical Features:", len(categorical_features))

**Check Missing Values**

In [ ]:
print("\n========== MISSING VALUES ==========")

missing_values = X.isnull().sum()

print(missing_values[missing_values > 0])

print("\nTotal Missing Values:", X.isnull().sum().sum())
print("Target Missing Values:", y.isnull().sum())

**Create Preprocessing Pipeline**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing Pipeline Created Successfully!")

**Final ML Dataset Check**

In [ ]:
print("\n========================================")
print("       FINAL ML DATA CHECK")
print("========================================")

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nX Missing Values:", X.isnull().sum().sum())
print("y Missing Values:", y.isnull().sum())

print("\nTarget Statistics:")
display(y.describe())

**Important Leakage Check**

In [ ]:
# These features depend directly on sold_quantity
leakage_features = [
    "inventory_utilization",
    "inventory_utilization_pct",
    "review_engagement"
]

print("\nExcluded Leakage Features:")
display(leakage_features)

**Train/Test Split**

In [ ]:
# Display all columns
display("Model DataFrame Columns:")
display(model_df.columns.tolist())

In [ ]:
print("Model DataFrame Shape:", model_df.shape)

display(model_df.head())

In [ ]:
# ========================================
# CHECK MODEL DATASET COLUMNS
# ========================================

print("model_df Shape:", model_df.shape)

print("\nmodel_df Columns:")
for i, col in enumerate(model_df.columns, 1):
    print(i, "→", col)

In [ ]:
print("\nFirst 5 Rows:")
display(model_df.head())

In [ ]:
target = "Item_Outlet_Sales"

In [ ]:
print(model_df.columns.tolist())

In [ ]:
# Display available columns
print("Available Columns:")
print(model_df.columns.tolist())

In [ ]:
# Display numeric columns
numeric_columns = model_df.select_dtypes(include="number").columns.tolist()

print("\nNumeric Columns:")
print(numeric_columns)

In [ ]:
# Select target
target = numeric_columns[-1]

print("Selected Target:", target)

# Features and target
X = model_df.drop(columns=[target])
y = model_df[target]

print("\nX Shape:", X.shape)
print("y Shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\n========================================")
print("        TRAIN / TEST SPLIT")
print("========================================")

print("X_train Shape:", X_train.shape)
print("X_test Shape :", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape :", y_test.shape)

In [ ]:
print("\nTraining Samples:", len(X_train))
print("Testing Samples :", len(X_test))

print("\nTraining Percentage:",
      round(len(X_train) / len(X) * 100, 2), "%")

display("Testing Percentage:",
      round(len(X_test) / len(X) * 100, 2), "%")

In [ ]:
import pandas as pd

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Samples": [len(X_train), len(X_test)],
    "Percentage": [
        round(len(X_train) / len(X) * 100, 2),
        round(len(X_test) / len(X) * 100, 2)
    ]
})

display(split_summary)

**Model Training**

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

# Copy training and testing data
X_train = X_train.copy()
X_test = X_test.copy()

# Convert categorical columns into numerical values
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    encoder = LabelEncoder()

    # Fit on combined train + test values
    combined_values = pd.concat([X_train[col], X_test[col]]).astype(str)
    encoder.fit(combined_values)

    X_train[col] = encoder.transform(X_train[col].astype(str))
    X_test[col] = encoder.transform(X_test[col].astype(str))

# Drop any datetime columns that cannot be directly handled by sklearn
datetime_cols = X_train.select_dtypes(include=["datetime64", "datetime64[ns]"]).columns
X_train = X_train.drop(columns=datetime_cols, errors='ignore')
X_test = X_test.drop(columns=datetime_cols, errors='ignore')

# Train Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Train Random Forest Regression
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)
rf_model.fit(X_train, y_train)

# Display training status
display("========================================")
display("        MODEL TRAINING COMPLETED")
display("========================================")

display("Linear Regression: Trained Successfully")
display("Random Forest Regression: Trained Successfully")

display("Number of Training Samples:", len(X_train))
display("Number of Features:", X_train.shape[1])

In [ ]:
def generate_model_predictions_code(model_var_name="rf_model", test_features="X_test", test_target="y_test"):
    """
    Generates and returns ready-to-execute Python code for making model predictions,
    calculating evaluation metrics (MAE, MSE, RMSE, R²), and plotting performance.
    """
    code_template = f"""import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Make predictions
y_pred = {model_var_name}.predict({test_features})

# 2. Calculate Regression Metrics
mae = mean_absolute_error({test_target}, y_pred)
mse = mean_squared_error({test_target}, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score({test_target}, y_pred)

print("========================================")
print("       MODEL EVALUATION METRICS")
print("========================================")
print(f"Mean Absolute Error (MAE)  : {{mae:.4f}}")
print(f"Mean Squared Error (MSE)   : {{mse:.4f}}")
print(f"Root Mean Squared Error (RMSE): {{rmse:.4f}}")
print(f"R-squared Score (R²)       : {{r2:.4f}}")
print("========================================")

# 3. Plot Actual vs Predicted
plt.figure(figsize=(10, 6))
sns.scatterplot(x={test_target}, y=y_pred, alpha=0.5, color='teal')

# Add baseline diagonal line representing perfect accuracy
min_val = min({test_target}.min(), y_pred.min())
max_val = max({test_target}.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Fit')

plt.title("Actual vs. Predicted Values ({model_var_name})", fontsize=14, fontweight='bold')
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
"""
    return code_template

# Generate model evaluation code snippet dynamically
generated_code = generate_model_predictions_code(model_var_name="rf_model")
print("Generated Code Snippet:\n")
print(generated_code)

**Model Evaluation**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ========================================
# MODEL PREDICTIONS
# ========================================

linear_pred = linear_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

# ========================================
# LINEAR REGRESSION METRICS
# ========================================

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_mse = mean_squared_error(y_test, linear_pred)
linear_rmse = np.sqrt(linear_mse)
linear_r2 = r2_score(y_test, linear_pred)

# ========================================
# RANDOM FOREST METRICS
# ========================================

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

# ========================================
# DISPLAY MODEL EVALUATION
# ========================================

display("========================================")
display("       MODEL EVALUATION METRICS")
display("========================================")

display(
    pd.DataFrame({
        "Model": [
            "Linear Regression",
            "Random Forest Regression"
        ],
        "MAE": [
            round(linear_mae, 4),
            round(rf_mae, 4)
        ],
        "MSE": [
            round(linear_mse, 4),
            round(rf_mse, 4)
        ],
        "RMSE": [
            round(linear_rmse, 4),
            round(rf_rmse, 4)
        ],
        "R² Score": [
            round(linear_r2, 4),
            round(rf_r2, 4)
        ]
    })
)

**Actual vs Predicted — Random Forest**

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    x=y_test,
    y=rf_pred,
    alpha=0.6
)

min_val = min(y_test.min(), rf_pred.min())
max_val = max(y_test.max(), rf_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--",
    label="Perfect Prediction"
)

plt.title(
    "Actual vs Predicted Values — Random Forest Regression",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Prediction**

In [ ]:
# ========================================
# STEP 9 — MODEL PREDICTION
# ========================================

# Use Random Forest as the final model
final_model = rf_model

# Generate predictions
y_pred = final_model.predict(X_test)

# Create prediction comparison table
prediction_results = pd.DataFrame({
    "Actual Value": y_test.values,
    "Predicted Value": y_pred
})

# Calculate prediction error
prediction_results["Error"] = (
    prediction_results["Actual Value"]
    - prediction_results["Predicted Value"]
)

# Display prediction results
display("          MODEL PREDICTION ")

display(prediction_results.head(20))

**Prediction Summary**

In [ ]:
# ========================================
# PREDICTION SUMMARY
# ========================================

display("========================================")
display("        PREDICTION SUMMARY")
display("========================================")

display("Total Predictions:", len(y_pred))
display("Average Actual Value:", round(y_test.mean(), 4))
display("Average Predicted Value:", round(y_pred.mean(), 4))
display("Average Prediction Error:",
        round(prediction_results["Error"].mean(), 4))

**Prediction Visualization**

In [ ]:
# ========================================
# ACTUAL VS PREDICTED — PREDICTION PLOT
# ========================================

plt.figure(figsize=(12, 6))

plt.plot(
    y_test.values,
    label="Actual Values",
    linewidth=2
)

plt.plot(
    y_pred,
    label="Predicted Values",
    linewidth=2
)

plt.title(
    "Actual vs Predicted Values",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Test Sample")
plt.ylabel("Target Value")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## **Final Project Report**

In [ ]:
# ========================================
# STEP 10 — FINAL PROJECT REPORT
# ========================================

display("""
========================================================
              MACHINE LEARNING PROJECT REPORT
========================================================

Project: Blinkit Dataset Prediction

PROJECT WORKFLOW
--------------------------------------------------------
1. Data Collection
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Machine Learning Preparation
6. Train/Test Split
7. Model Training
8. Model Evaluation
9. Prediction
10. Final Project Report

MODELS USED
--------------------------------------------------------
• Linear Regression
• Random Forest Regression

EVALUATION METRICS
--------------------------------------------------------
• Mean Absolute Error (MAE)
• Mean Squared Error (MSE)
• Root Mean Squared Error (RMSE)
• R² Score

FINAL MODEL
--------------------------------------------------------
Random Forest Regression

========================================================
              PROJECT COMPLETED SUCCESSFULLY
========================================================
""")

**Final Model Performance**

In [ ]:
final_report = pd.DataFrame({
    "Metric": [
        "MAE",
        "MSE",
        "RMSE",
        "R² Score"
    ],
    "Linear Regression": [
        round(linear_mae, 4),
        round(linear_mse, 4),
        round(linear_rmse, 4),
        round(linear_r2, 4)
    ],
    "Random Forest Regression": [
        round(rf_mae, 4),
        round(rf_mse, 4),
        round(rf_rmse, 4),
        round(rf_r2, 4)
    ]
})

display("FINAL MODEL PERFORMANCE")
display(final_report)

**Best Model Selection**

In [ ]:
# Select the model with the highest R² score
if rf_r2 >= linear_r2:
    best_model_name = "Random Forest Regression"
    best_r2 = rf_r2
else:
    best_model_name = "Linear Regression"
    best_r2 = linear_r2

display("========================================")
display("          BEST MODEL")
display("========================================")

display("Best Model:", best_model_name)
display("Best R² Score:", round(best_r2, 4))

## **Feature Importance & Model Insights**

In [ ]:
# ========================================
# STEP 11 — FEATURE IMPORTANCE
# ========================================

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

# Sort by importance
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Display feature importance
display("========================================")
display("         FEATURE IMPORTANCE")
display("========================================")

display(feature_importance)

**Feature Importance Plot**

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)

plt.title(
    "Top 15 Feature Importance — Random Forest",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

**Top 10 Important Features**

In [ ]:
display("========================================")
display("       TOP 10 IMPORTANT FEATURES")
display("========================================")

display(feature_importance.head(10))

**Final Model Visualization & Business Insights**

In [ ]:
# ========================================
# STEP 12 — FINAL MODEL PERFORMANCE
# ========================================

display("========================================")
display("       FINAL MODEL PERFORMANCE")
display("========================================")

display("Final Model:", best_model_name)
display("R² Score:", round(best_r2, 4))
display("MAE:", round(rf_mae, 4))
display("RMSE:", round(rf_rmse, 4))

In [ ]:
# ========================================
# MOST IMPORTANT FEATURE
# ========================================

top_feature = feature_importance.iloc[0]["Feature"]
top_importance = feature_importance.iloc[0]["Importance"]

display("========================================")
display("        TOP IMPORTANT FEATURE")
display("========================================")

display("Feature:", top_feature)
display("Importance:", round(top_importance, 4))

In [ ]:
# ========================================
# FINAL ACTUAL VS PREDICTED PLOT
# ========================================

plt.figure(figsize=(10, 7))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.6
)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    linewidth=2,
    label="Perfect Prediction"
)

plt.title(
    "Random Forest — Actual vs Predicted",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Actual Values", fontsize=12)
plt.ylabel("Predicted Values", fontsize=12)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# FINAL BUSINESS INSIGHTS
# ========================================

display("""
========================================================
                 FINAL BUSINESS INSIGHTS
========================================================

• The Random Forest Regression model was evaluated
  against the Linear Regression model.

• Model performance was measured using MAE, MSE,
  RMSE and R² Score.

• Feature importance analysis identifies the variables
  that contribute most strongly to predictions.

• Actual vs Predicted analysis helps determine how
  closely the model follows the real target values.

• The final model can be used to generate predictions
  for new unseen data.

========================================================
              MACHINE LEARNING ANALYSIS COMPLETE
========================================================
""")

## **Hyperparameter Tuning**

In [ ]:
# ========================================
# STEP 13 — HYPERPARAMETER TUNING
# ========================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Parameter grid
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

# Base model
rf_base = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# Grid Search
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=0
)

# Train Grid Search
grid_search.fit(X_train, y_train)

# Best model
tuned_rf_model = grid_search.best_estimator_

# Display results
display("========================================")
display("       HYPERPARAMETER TUNING")
display("========================================")

display("Best Parameters:")
display(grid_search.best_params_)

display("Best Cross-Validation R² Score:",
        round(grid_search.best_score_, 4))

In [ ]:
# ========================================
# TUNED MODEL EVALUATION
# ========================================

tuned_pred = tuned_rf_model.predict(X_test)

tuned_mae = mean_absolute_error(y_test, tuned_pred)
tuned_mse = mean_squared_error(y_test, tuned_pred)
tuned_rmse = np.sqrt(tuned_mse)
tuned_r2 = r2_score(y_test, tuned_pred)

display("========================================")
display("       TUNED MODEL PERFORMANCE")
display("========================================")

display("MAE:", round(tuned_mae, 4))
display("MSE:", round(tuned_mse, 4))
display("RMSE:", round(tuned_rmse, 4))
display("R² Score:", round(tuned_r2, 4))

In [ ]:
# ========================================
# BEFORE vs AFTER TUNING
# ========================================

tuning_comparison = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R² Score"],
    "Original Random Forest": [
        rf_mae,
        rf_mse,
        rf_rmse,
        rf_r2
    ],
    "Tuned Random Forest": [
        tuned_mae,
        tuned_mse,
        tuned_rmse,
        tuned_r2
    ]
})

tuning_comparison = tuning_comparison.round(4)

display("========================================")
display("      BEFORE vs AFTER TUNING")
display("========================================")

display(tuning_comparison)

In [ ]:
# ========================================
# SELECT FINAL MODEL
# ========================================

if tuned_r2 >= rf_r2:
    final_model = tuned_rf_model
    final_model_name = "Tuned Random Forest Regression"
    final_r2 = tuned_r2
else:
    final_model = rf_model
    final_model_name = "Original Random Forest Regression"
    final_r2 = rf_r2

display("========================================")
display("          FINAL MODEL SELECTED")
display("========================================")

display("Final Model:", final_model_name)
display("Final R² Score:", round(final_r2, 4))

### **Final Prediction Using Optimized Model**

In [ ]:
# ========================================
# STEP 14 — FINAL PREDICTION
# ========================================

# Generate predictions using final model
final_predictions = final_model.predict(X_test)

# Create final prediction dataframe
final_prediction_df = pd.DataFrame({
    "Actual Value": y_test.values,
    "Predicted Value": final_predictions
})

# Calculate prediction error
final_prediction_df["Error"] = (
    final_prediction_df["Actual Value"]
    - final_prediction_df["Predicted Value"]
)

# Absolute error
final_prediction_df["Absolute Error"] = (
    final_prediction_df["Error"].abs()
)

display("========================================")
display("          FINAL PREDICTIONS")
display("========================================")

display(final_prediction_df.head(20))

In [ ]:
# ========================================
# FINAL PREDICTION METRICS
# ========================================

final_mae = mean_absolute_error(
    y_test,
    final_predictions
)

final_mse = mean_squared_error(
    y_test,
    final_predictions
)

final_rmse = np.sqrt(final_mse)

final_r2 = r2_score(
    y_test,
    final_predictions
)

display("========================================")
display("       FINAL MODEL METRICS")
display("========================================")

display("Model:", final_model_name)
display("MAE:", round(final_mae, 4))
display("MSE:", round(final_mse, 4))
display("RMSE:", round(final_rmse, 4))
display("R² Score:", round(final_r2, 4))

In [ ]:
# ========================================
# FINAL ACTUAL VS PREDICTED
# ========================================

plt.figure(figsize=(10, 7))

plt.scatter(
    y_test,
    final_predictions,
    alpha=0.6
)

min_value = min(
    y_test.min(),
    final_predictions.min()
)

max_value = max(
    y_test.max(),
    final_predictions.max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    linewidth=2,
    label="Perfect Prediction"
)

plt.title(
    f"Actual vs Predicted — {final_model_name}",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# FINAL PREDICTION TREND
# ========================================

plt.figure(figsize=(12, 6))

plt.plot(
    y_test.values,
    label="Actual Values",
    linewidth=2
)

plt.plot(
    final_predictions,
    label="Predicted Values",
    linewidth=2
)

plt.title(
    "Final Model — Actual vs Predicted Trend",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Test Sample")
plt.ylabel("Target Value")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## **Project Conclusion & Summary**

In [ ]:
# ========================================
# STEP 15 — FINAL PROJECT CONCLUSION
# ========================================

display("""
========================================================
             FINAL PROJECT CONCLUSION
========================================================

PROJECT:
Blinkit Dataset — Machine Learning Prediction

DATA ANALYSIS:
• Data Cleaning
• Exploratory Data Analysis (EDA)
• Feature Engineering
• Feature Preparation

MACHINE LEARNING:
• Train/Test Split
• Linear Regression
• Random Forest Regression
• Hyperparameter Tuning

MODEL EVALUATION:
• MAE
• MSE
• RMSE
• R² Score

FINAL MODEL:
Optimized Random Forest Regression

PREDICTION:
The final model was used to generate predictions
on unseen test data.

VISUALIZATION:
• Actual vs Predicted Values
• Prediction Trend
• Feature Importance
• Model Performance Comparison

========================================================
              PROJECT SUCCESSFULLY COMPLETED
========================================================
""")

In [ ]:
# ========================================
# FINAL MODEL REPORT
# ========================================

final_model_report = pd.DataFrame({
    "Project": ["Blinkit Dataset Prediction"],
    "Final Model": [final_model_name],
    "MAE": [round(final_mae, 4)],
    "MSE": [round(final_mse, 4)],
    "RMSE": [round(final_rmse, 4)],
    "R² Score": [round(final_r2, 4)]
})

display("FINAL MODEL REPORT")
display(final_model_report)